In [1]:
!mv /kaggle/working/t2i_run /kaggle/working/t2i_run_flickr8k

mv: cannot stat '/kaggle/working/t2i_run': No such file or directory


In [2]:
!find /kaggle/input -maxdepth 6 \( -name "*.csv" -o -name "*.token*" -o -name "captions*" \) | head
!find /kaggle/input -maxdepth 6 -name "*.jpg" | head -3


^C
/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images/2715746315.jpg
/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images/3463034205.jpg
/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images/268704620.jpg
find: ‘standard output’: Broken pipe
find: write error


In [4]:
import tensorflow as tf
v = ".".join(tf.__version__.split(".")[:2])
!pip install -q "tf-keras~={v}.0" sentence-transformers huggingface_hub
import os, shutil, re
src = "/kaggle/input/datasets/geeteshsin/t2i-code"
dst = "/kaggle/working/code"
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(src, dst)
p = dst + "/train_kaggle.py"
s = open(p).read()

new_ds = '''def make_ds(images, pair_img, embs, lo, hi, batch, training):
    n = hi - lo

    def gen():
        rng = np.random.default_rng()
        while True:
            order = (rng.permutation(n) + lo) if training else np.arange(lo, hi)
            for i in order:
                yield images[pair_img[i]], embs[i]
            if not training:
                return

    sig = (tf.TensorSpec((IMAGE_SIZE, IMAGE_SIZE, 3), tf.uint8),
           tf.TensorSpec(tuple(embs.shape[1:]), tf.float16))
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)

    def load(img, emb):
        img = tf.cast(img, tf.float32) / 127.5 - 1.0
        emb = tf.cast(emb, tf.float32)
        if training:
            img = tf.image.random_flip_left_right(img)
            emb = tf.where(tf.random.uniform([]) < TEXT_DROPOUT, tf.zeros_like(emb), emb)
        return img, emb

    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch, drop_remainder=True).prefetch(tf.data.AUTOTUNE)


'''
pat = re.compile(r"def make_ds\(.*?(?=# -{10,} strategy)", re.S)
assert pat.search(s)
s = pat.sub(lambda m: new_ds, s, 1)

parser = '''def parse_caps_any(path):
    import csv
    out = {}
    with open(path, encoding="utf-8", errors="ignore") as f:
        first = f.readline()
        f.seek(0)
        if "|" in first:
            for line in f:
                r = line.rstrip("\\n").split("|", 2)
                if len(r) == 3 and r[1].strip().isdigit() and r[2].strip():
                    out.setdefault(r[0].strip(), []).append(r[2].strip())
        elif "\\t" in first:
            for line in f:
                r = line.rstrip("\\n").split("\\t", 1)
                if len(r) == 2 and r[1].strip():
                    out.setdefault(r[0].split("#")[0].strip(), []).append(r[1].strip())
        else:
            for r in csv.reader(f):
                if len(r) >= 2 and not r[0].lower().startswith("image"):
                    c = ",".join(r[1:]).strip()
                    if c:
                        out.setdefault(r[0].strip(), []).append(c)
    return out


'''
s = s.replace("def build_arrays(persist):", parser + "def build_arrays(persist):", 1)

a = "    cap_file = find_flickr()\n"
b = '    img_dir = cap_file.parent / "Images"\n    img_dir = img_dir if img_dir.exists() else cap_file.parent\n'
c = "    caps = parse_captions_file(cap_file)\n"
for x in (a, b, c):
    assert x in s
s = s.replace(a, '    cap_file = Path(os.environ["T2I_CAPTIONS"]) if os.environ.get("T2I_CAPTIONS") else find_flickr()\n', 1)
s = s.replace(b, '    if os.environ.get("T2I_IMAGES"):\n        img_dir = Path(os.environ["T2I_IMAGES"])\n    else:\n        img_dir = cap_file.parent / "Images"\n        img_dir = img_dir if img_dir.exists() else cap_file.parent\n', 1)
s = s.replace(c, "    caps = parse_caps_any(cap_file)\n", 1)
open(p, "w").write(s)
print("patched")

patched


In [5]:
import os
os.environ["T2I_SMOKE"] = "1"
os.environ["T2I_FAKE_DATA"] = "1"
os.environ["T2I_IMAGE_SIZE"] = "32"
os.environ["T2I_BASE_FILTERS"] = "8"
!python /kaggle/working/code/train_kaggle.py

I0000 00:00:1793533163.435862     158 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793533163.440118     158 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
[2026-11-01 11:39:23] No TPU (NotFoundError); checking GPUs.
[2026-11-01 11:39:23] ======================================================================
[2026-11-01 11:39:23] Device: GPU, replicas 2, global batch 64, peak LR 2.83e-04, session limit 0.0 h
[2026-11-01 11:39:23] Data: 48 images, 144 pairs (train 129); target 150 passes = 19,350 samples
[2026-11-01 11:39:32] Model built: 714,859 parameters
[2026-11-01 11:39:40] Fresh run from random init.
[2026-11-01 11:39:40] Progress: step 0, 0/19,350 samples (0.0 passes)
I0000 00:00:1793533230.777895  

In [6]:
import os
for k in ["T2I_SMOKE", "T2I_FAKE_DATA", "T2I_IMAGE_SIZE", "T2I_BASE_FILTERS"]:
    os.environ.pop(k, None)
!rm -rf /kaggle/working/t2i_run

In [ ]:
import os
os.environ["HF_REPO"] = "geeteshcodes/t2i-flickr30k"
os.environ["T2I_CAPTIONS"] = "/kaggle/input/datasets/eeshawn/flickr30k/captions.txt"
os.environ["T2I_IMAGES"] = "/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images"
os.environ["T2I_BASE_FILTERS"] = "48"
os.environ["T2I_TARGET_PASSES"] = "15"
!python /kaggle/working/code/train_kaggle.py

I0000 00:00:1793534897.433730    6287 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793534897.436005    6287 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
[2026-11-01 12:08:17] No TPU (NotFoundError); checking GPUs.
[2026-11-01 12:08:17] ======================================================================
[2026-11-01 12:08:17] Device: GPU, replicas 2, global batch 64, peak LR 2.83e-04, session limit 11.0 h
[2026-11-01 12:08:18] Hugging Face sync enabled: geeteshcodes/t2i-flickr30k
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 0 files: 0it [00:00, ?it/s]
Download complete: :                                        |  0.00B            
Download complete: :     